# AP02 Spick – Fehleranalyse-Checkliste

**Für AP02: Data Ingestion, Data Preparation und Exploratory Data Analysis (EDA).** Zuerst die zehn Kernfragen prüfen; nur bei Bedarf die thematischen Ergänzungen verwenden. Die Beispiele sind kurze Nachschlagehinweise, keine vollständigen Lösungen. **Alle Zellen sind Markdown – kein Setup nötig.**

**Prüfungsregel:** Immer zuerst die verlangte Operation identifizieren (z. B. `qcut` statt `cut`, Spearman statt Pearson, einfacher statt gruppierter Scatter Plot). Danach Daten/Spalten, Datentypen und Ergebnis überprüfen. Namen wie `df`, `spalte` oder `schluessel` sind Beispiele und müssen zur Aufgabe passen.

## Die zehn Kernfragen

1. **Datei nicht gefunden (`FileNotFoundError`)?** → `Path.cwd()` zeigt das Arbeitsverzeichnis; `Path.cwd().rglob('*.json')` sucht Dateien darunter. Relative Pfade beziehen sich auf den **Jupyter-Kernel**, nicht zwingend auf den Notebook-Ordner. `Path('datei.json').is_file()` prüft die Existenz.
2. **Spalte fehlt (`KeyError`)?** → `df.columns.tolist()` und `df.head()` prüfen. Schreibweise, Leerzeichen und Gross-/Kleinschreibung beachten. Nach `df[['a', 'b']]` fehlen alle anderen Spalten im neuen Teil-DataFrame; nach `merge()` ist eine Spalte möglicherweise nicht übernommen worden.
3. **Zahlen werden als Text behandelt oder falsch extrahiert?** → `df.dtypes` prüfen. `pd.to_numeric(..., errors='coerce')` wandelt konvertierbare Texte in Zahlen um. **CHF 12'990 darf nicht als 12 interpretiert werden**; bei Ganzzahlpreisen Tausendertrennzeichen korrekt entfernen bzw. alle Zifferngruppen zusammenfügen.
4. **Falsche Zeilenanzahl oder unerwartete `NaN`?** → `df.shape`, `df.isna().sum()` und `df.duplicated().sum()` prüfen. `dropna()` entfernt Zeilen mit mindestens einem fehlenden Wert, `dropna(how='all')` nur vollständig leere Zeilen. `drop_duplicates()` entfernt Duplikate nur nach gewählten Kriterien.
5. **`merge()` liefert zu wenige/zu viele Datensätze?** → Gemeinsamen Schlüssel (`on='product_id'` bzw. `on='offer_id'`), Schlüsseltypen, Duplikate und `how` prüfen. Standard ist `inner` (nur Treffer); `left` behält alle linken Zeilen. Ein mehrfach vorkommender Schlüssel kann Zeilen vervielfachen.
6. **HTML-Element oder Tabelle nicht gefunden (`NoneType`)?** → Existiert das gesuchte `id="..."` oder `class="..."`? `soup.find('table', class_='spec-table')` sucht nach einer **Klasse**, `id=` nach einer ID. `soup.find_all('table')` zur Kontrolle; `find()` kann `None` zurückgeben.
7. **Falsche Klassen / Diskretisierung?** → `pd.cut(..., bins=3)` erzeugt **gleich breite Werteintervalle**; `pd.qcut(..., q=3)` erzeugt **ungefähr gleich grosse Datensatzgruppen**. Mit `df['segment'] = ...` entsteht die neue Spalte; `labels` und Anzahl Klassen müssen übereinstimmen.
8. **Unplausible Kennzahlen oder ein extrem hoher Wert?** → `df['spalte'].describe()`, `sort_values()`, Boxplot bzw. IQR nutzen. **Ausreisser ist nicht gleich Eingabefehler**: zuerst fachlich plausibilisieren, dann begründet behandeln. Durchschnitt reagiert stärker auf Ausreisser als Median.
9. **Falsches Diagramm?** → Einfache Beziehung zweier numerischer Variablen: `ax.scatter(df['x'], df['y'])`. `groupby()` und `legend()` **nur bei verlangter Aufteilung nach Kategorien**. X-/Y-Spalten und Achsenbeschriftungen müssen zur Aufgabe passen.
10. **Falsche Korrelation oder Interpretation?** → `corr(method='pearson')` misst **lineare**, `corr(method='spearman')` **monotone Rangzusammenhänge**. Für genau zwei Spalten: `df['x'].corr(df['y'], method='spearman')`; auf zwei Stellen: `round(..., 2)`. Korrelation bedeutet **nicht** Kausalität.

## Data Ingestion

11. **CSV wird in einer einzigen Spalte geladen?** → Falsches Trennzeichen: `pd.read_csv('datei.csv', sep=';')` bei Semikolon-Dateien. Umlaute fehlerhaft? `encoding` prüfen. Dezimal-/Tausendertrennzeichen bei Bedarf ausdrücklich behandeln.
12. **JSON-Struktur passt nicht zum erwarteten DataFrame?** → `df.head()`, `df.columns.tolist()` und `type(...)` prüfen. Eine JSON-Liste von Objekten unterscheidet sich von verschachtelten JSON-Daten; bei API-Antworten ggf. relevante Teilliste auswählen oder mit `pd.json_normalize()` normalisieren.
13. **HTML-Tabellenüberschriften erscheinen als leere Zeilen?** → Bei `row.find_all('td')` werden `<th>`-Zellen nicht gelesen. Zeilen mit `<th>`-Zwischenüberschriften können dadurch zu `NaN`/`NaN` werden; **nach** der DataFrame-Erstellung `dropna(how='all').reset_index(drop=True)` verwenden. `dropna()` würde zusätzlich teilgefüllte Zeilen entfernen.
14. **Tabellenzeilen haben unterschiedlich viele Zellen?** → `<tr>` sind Zeilen, `<td>` Datenzellen, `<th>` Überschriften. Vor dem `pd.DataFrame(rows, columns=[...])` die Listenlängen prüfen; ein Filter wie `len(row) == 2` ist **nur korrekt, wenn genau zwei Datenzellen erwartet werden**. `colspan` und Zeilengruppen beachten.
15. **BeautifulSoup extrahiert keinen oder falschen Text?** → Tag, CSS-Klasse und Eltern-/Kindelement prüfen. `find()` liefert **ein** Element oder `None`; `find_all()` liefert eine **Liste**. Text: `.get_text(strip=True)`; Attribute: `.get('href')` usw. Keinen `None.get_text()`-Aufruf machen.
16. **API liefert keine nutzbaren Daten?** → URL, HTTP-Methode (GET/POST), `params`/`data` und Statuscode kontrollieren; bei Requests `response.raise_for_status()` und erst dann `response.json()`. Authentisierung, Rate Limits (z. B. 429), Zeitüberschreitung sowie leere Antworten berücksichtigen.
17. **Selenium findet ein Element nicht?** → Wird die Seite dynamisch geladen? Selektor/XPath und tatsächlichen DOM prüfen; ggf. `WebDriverWait(...).until(...)` statt sofortigem Zugriff. Bei Browserstartfehlern Browser/WebDriver-Verfügbarkeit kontrollieren. Ein Selenium-Selektor ist kein BeautifulSoup-Selektor.
18. **RSS-Feed enthält keine erwarteten Beiträge?** → Feed-URL, Parser-Ergebnis (`feed.entries`) und tatsächlich vorhandene Felder prüfen; nicht jeder Eintrag hat z. B. `published` oder `summary`. Unterschied RSS/XML vs. API/JSON beachten.

## Data Preparation

19. **Bereinigung entfernt zu viele Zeilen?** → `dropna()` entfernt schon bei **einem** fehlenden Wert. Mit `subset=['wichtige_spalte']` gezielt entscheiden; `dropna(how='all')` entfernt nur vollständig leere Zeilen. Vorher/nachher `df.shape` vergleichen.
20. **`SettingWithCopyWarning` oder Änderungen bleiben aus?** → Bei einem Teil-DataFrame explizit `.copy()` verwenden: `df_sub = df[['a', 'b']].copy()`. Spalten im ursprünglichen DataFrame gezielt mit `df.loc[maske, 'spalte'] = ...` setzen.
21. **Werte stehen nach Zuweisung in falschen Zeilen oder werden `NaN`?** → pandas richtet `Series` bei Zuweisungen **über Indexlabels** aus. Nach `drop_duplicates()`/Filtern bei Bedarf `.reset_index(drop=True)` nutzen oder passende Indizes erhalten; nicht blind eine neue `pd.Series(...)` mit Index 0,1,2 zuweisen.
22. **Filter liefert das Gegenteil oder nichts?** → Vergleichsoperatoren, Datentypen, Klammern und `&` (UND), `|` (ODER), `~` (NICHT) kontrollieren. Beispiel: `df.loc[df['price_chf'] != 12990]` schliesst **genau diesen Preis** aus; `df['spalte'].isin([...])` prüft Zugehörigkeit.
23. **`merge()` erzeugt Spalten `_x` und `_y`?** → Beide DataFrames enthalten gleichnamige **Nicht-Schlüsselspalten**. Vorher nur benötigte Spalten auswählen oder `suffixes=('_links', '_rechts')` setzen. Es werden **nur explizit ausgewählte Spalten** aus einem Teil-DataFrame übernommen.
24. **`qcut()` meldet „Bin edges must be unique“?** → Viele gleiche Werte verhindern eindeutige Quantilgrenzen. Anzahl eindeutiger Werte prüfen; `duplicates='drop'` kann die **Anzahl der Klassen reduzieren**, sodass Labels neu angepasst werden müssen – nicht ungeprüft einsetzen.
25. **Neue Spalte fehlt oder wurde überschrieben?** → `df['neue_spalte'] = ...` legt die Spalte an, überschreibt sie aber bei erneutem Ausführen. `df[['bestehend', 'neu']]` wählt nur bereits existierende Spalten aus; es **erstellt keine**.
26. **Kategoriewerte werden unerwartet `NaN`?** → Bei `pd.cut()` mit **selbst definierten** `bins` müssen Werte im Intervall liegen; Grenzwerte und `include_lowest`/`right` prüfen. Bei One-Hot Encoding entstandene Spalten mit `df.columns` kontrollieren.
27. **Gruppenzahlen stimmen nicht?** → `groupby(...).size()` zählt **Zeilen** pro Gruppe; `groupby(...)[spalte].count()` zählt nur **nicht fehlende Werte**. `mean()` vs. `median()` anhand der geforderten Kennzahl wählen. `reset_index()` nur, wenn Gruppenschlüssel wieder reguläre Spalten sein sollen.
28. **Kombinierte Tabellen haben unerwartet doppelte Zeilen/Spalten?** → `pd.concat(..., axis=0)` hängt **Zeilen** an; `axis=1` verbindet **Spalten nach Indexlabels**. `merge()` verknüpft dagegen über Schlüssel. Für `pivot()` benötigen Index-/Spaltenkombinationen eindeutige Werte; `pivot_table()` kann Duplikate aggregieren.
29. **Skalierung/Transformation erzeugt unerwartete Werte?** → Prüfen, ob die gewählte Methode verlangt wird (Min-Max, Z-Score, Robust Scaling, Log, Wurzel). Für `log()` nur positive Werte, für `sqrt()` keine negativen reellen Werte; `MinMaxScaler` ist nicht `StandardScaler`. Transformation nicht mit Datentypumwandlung verwechseln.

## EDA & Visualisierung

30. **Falsche Statistik für die Aufgabe?** → `describe()` liefert Standardübersicht, `.mean()` Durchschnitt, `.median()` Median, `.mode()` Modus, `.std()` Standardabweichung, `.quantile(...)` Quantile. **`50%` ist der Median**, nicht der Mittelwert. `round(2)` rundet nur die Darstellung/Berechnungsausgabe.
31. **Histogramm, Boxplot, Balkendiagramm und Scatter Plot verwechselt?** → **Histogramm:** Verteilung einer numerischen Variable; **Boxplot:** Median, Quartile und potenzielle Ausreisser; **Balkendiagramm/Countplot:** kategoriale Häufigkeiten; **Scatter Plot:** Beziehung zwischen zwei numerischen Variablen.
32. **Scatter Plot steht auf der falschen X-Achse?** → Die in der Aufgabe genannten numerischen Variablen plotten, z. B. `ax.scatter(df['price_chf'], df['avg_score'])`; nicht versehentlich `segment` statt `price_chf` verwenden. `set_xlabel`, `set_ylabel` und `set_title` passend beschriften.
33. **Gruppierte Grafik unerwartet leer oder `KeyError`?** → `groupby('power_type')` nur bei tatsächlich vorhandener kategorialer Spalte **und** wenn Gruppen/Farben verlangt sind. Nach `df[['brand', 'price_chf']]` gibt es dort z. B. **keine** Spalte `segment`, bevor sie ergänzt wird.
34. **Korrelationsmatrix statt eines einzelnen Korrelationswerts?** → `df[['x', 'y']].corr(method='spearman')` gibt eine **2×2-Matrix**; `df['x'].corr(df['y'], method='spearman')` einen **einzelnen Koeffizienten**. Für Pearson `method='pearson'` verwenden; der Standardwert ist Pearson.
35. **Korrelation verändert sich stark durch Ausreisser?** → Pearson reagiert besonders auf extreme Werte; Spearman arbeitet mit Rangplätzen und ist oft robuster. Wenn ein falscher Wert **nur für die Visualisierung** ausgeschlossen wird, nicht unbemerkt auch den Datensatz für die Korrelation ändern.
36. **Kreuztabelle stimmt nicht?** → `pd.crosstab(df['brand'], df['segment'])`: **erstes Argument = Zeilen**, **zweites = Spalten**. `margins=True` ergänzt Gesamtsummen; nur aktivieren, wenn sinnvoll/gefragt.
37. **Boxplot/Kategorien sind falsch sortiert oder fehlen?** → Bei Seaborn `x`/`y`, `data` und das optionale `order=[...]` kontrollieren. **Fest codierte Kategorien aus dem Spick** müssen zum neuen Datensatz passen; `order` bei Bedarf weglassen.
38. **Unplausibles Diagramm trotz fehlerfreier Ausführung?** → Datentypen, Achsenskalierung, Einheiten, Anzahl Beobachtungen, `NaN`, Duplikate und Ausreisser untersuchen. Ein Diagramm kann technisch korrekt und fachlich irreführend sein.
39. **Ergebnis wird zu stark interpretiert?** → Korrelation ≠ Kausalität. Positive/negative Korrelation beschreibt die **Richtung**, der Betrag die **Stärke**; bei Spearman bezieht sich der Zusammenhang auf die Rangfolge. Aus `r ≈ 0` folgt nicht generell „kein Zusammenhang“.
40. **Zeilen wurden bei Datenbereinigung entfernt: Nutzt die Grafik den richtigen DataFrame?** → Prüfen, ob `df_clean`, `df_merged` oder ein absichtlich gefilterter `df_plot` verwendet werden soll. Ausgabe und Grafiken müssen sich auf die verlangte Datenbasis beziehen.

## Fehlermeldungen – Kurzdiagnose

| Meldung / Symptom | Häufige Ursache | Erster Check |
|---|---|---|
| `FileNotFoundError` | Falscher Dateipfad / Arbeitsordner | `Path.cwd()`, `.rglob(...)`, `.is_file()` |
| `KeyError: 'spalte'` | Spalte fehlt oder wurde nicht übernommen | `df.columns.tolist()` |
| `NameError: name 'df' is not defined` | Zelle/Import nicht ausgeführt oder falscher Variablenname | Notebook-Reihenfolge und Variablen prüfen |
| `AttributeError: 'NoneType' ...` | `find()` hat kein HTML-Element gefunden | `id`/`class_` und `table is None` prüfen |
| `TypeError` bei Berechnung | Text statt Zahl oder inkompatible Typen | `df.dtypes`, `pd.to_numeric(...)` |
| `ValueError` bei `DataFrame(rows, columns=...)` | Anzahl Zeilenwerte passt nicht zur Spaltenanzahl | `rows[:5]`, `len(row)`, `columns` prüfen |
| `ValueError` bei `qcut()` | Nicht eindeutige Quantilgrenzen oder falsche Labelanzahl | Eindeutige Werte, `q`, `labels` kontrollieren |
| `SettingWithCopyWarning` | Veränderung an uneindeutigem DataFrame-Ausschnitt | `.copy()` oder `.loc[...]` |
| Unerwartet viele/wenige Zeilen | Duplikate, Join-Kardinalität, Filter, `dropna()` | `.shape` vor/nach jedem Schritt vergleichen |

## AP02-Prüfungsablauf zum Nachschlagen

1. **Aufgabenwortlaut lesen:** Ist `cut`/`qcut`, Pearson/Spearman, Mittelwert/Median, einfaches/gruppiertes Diagramm oder eine bestimmte Datenquelle **ausdrücklich** vorgegeben?
2. **Daten überprüfen:** Arbeitsverzeichnis/Dateipfad → `df.shape` → `df.columns.tolist()` → `df.dtypes` → `df.isna().sum()`.
3. **Schlüssel und Bereinigung prüfen:** Vor/nach `merge()`, `dropna()`, `drop_duplicates()` und Filtern die Zeilenzahl vergleichen; Indexausrichtung bei Series-Zuweisungen beachten.
4. **Aufgabe umsetzen:** Nur die geforderte Methode verwenden; `#ANPASSEN`-Werte (Datei, DataFrame, Spalten, Klassen, Labels, Plotachsen) aus dem Spick auf den aktuellen Datensatz übertragen.
5. **Ergebnis plausibilisieren:** Mit `head()`, `value_counts()`, `describe()`, einfachen Summen und passenden Diagrammen gegenprüfen. Statistische Ergebnisse nur aus den tatsächlich verwendeten Daten interpretieren.

**Merksatz:** Erst *richtige Datei und Spalten*, dann *richtige Methode*, zuletzt *richtige Datenbasis und Interpretation*. Eine fehlerfrei laufende Zelle ist noch keine korrekte Prüfungslösung.